## Q0. Setup

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"loaded {len(documents)} documents")   # should print 72

# We'll also need chunks for search later
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"{len(chunks)} chunks")

loaded 72 documents
295 chunks


In [2]:
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("BASE_URL"),
    http_client=httpx.Client(verify=False),
)

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
import json
from bacterio_compat import llm_structured

doc = documents[0]
user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
)

print(f"filename: {doc['filename']}\n")
for q in result.questions:
    print(f"  - {q}")
print(f"\ntokens: in={usage.prompt_tokens}, out={usage.completion_tokens}")

filename: 01-agentic-rag/lessons/01-intro.md

  - Why are we building this system from scratch instead of just using a framework?
  - How does a large language model actually work to generate text?
  - What are the main drawbacks or limitations of using LLMs on their own?
  - How does RAG help solve the problems that standard LLMs face?
  - What is the difference between the basic RAG pipeline in Part 1 and the agentic version in Part 2?

tokens: in=1196, out=925


## Q1. Generating questions.